# day-10-benchmarks — worked solutions & answer key

Solutions to the exercises in [`../lesson.ipynb`](../lesson.ipynb), plus the self-check answer key. **Try each exercise yourself first** — the value is in the attempt, not the answer.

In [10]:
# ---- Solution 1 ----
def normalize(score, n_options): return (score - 1/n_options) / (1 - 1/n_options)
per_smart = {s: np.mean([run_mmlu(smart_model)[1][s] for _ in range(300)]) for s in MMLU}
print("raw :", {k: round(v,2) for k,v in per_smart.items()})
print("norm:", {k: round(normalize(v,4),2) for k,v in per_smart.items()})
print("-> 'law' normalises near 0: the raw ~0.4 was almost entirely the 0.25 floor.")

raw : {'math': np.float64(0.92), 'history': np.float64(0.61), 'biology': np.float64(0.85), 'law': np.float64(0.38)}
norm: {'math': np.float64(0.9), 'history': np.float64(0.49), 'biology': np.float64(0.8), 'law': np.float64(0.17)}
-> 'law' normalises near 0: the raw ~0.4 was almost entirely the 0.25 floor.


In [11]:
# ---- Solution 2 ----
import math
pass10 = 0.80
p = 1 - (1 - pass10) ** (1/10)
print(f"S2: implied per-sample p = {p:.3f}  => pass@1 = {p:.3f}, pass@3 = {1-(1-p)**3:.3f}")

S2: implied per-sample p = 0.149  => pass@1 = 0.149, pass@3 = 0.383


In [12]:
# ---- Solution 4 ----
scores = {f: np.mean([run_mmlu(lambda su,q,o: formatted_model(su,q,o,f))[0] for _ in range(200)])
          for f in FORMAT_BONUS}
spread = max(scores.values()) - min(scores.values())
print(f"S4: format-only MMLU spread = {spread:.2f} ({spread*100:.0f} points)")
print("   a typical #1-vs-#5 leaderboard gap is ~2-4 points -> smaller than harness noise.")
print("   Conclusion: only compare models run through the SAME evaluation harness.")

S4: format-only MMLU spread = 0.14 (14 points)
   a typical #1-vs-#5 leaderboard gap is ~2-4 points -> smaller than harness noise.
   Conclusion: only compare models run through the SAME evaluation harness.


### Solutions 3, 5, 6 (worked)

**S3:** paraphrasing breaks any model keyed on exact question text. The lesson: a *robust* skill
model should match on meaning (embedding / normalized key); a *contaminated* model keyed on the
answer index still fails when options are shuffled even if it handles paraphrase. Test both
perturbations together — paraphrase *and* option-shuffle — to separate real skill from memory.

**S5:** if every test question describes an event/fact created *after* the training cutoff, the
model cannot have seen the question or its answer during training, so the score can't be
inflated by memorisation. Downside: post-cutoff data is scarce and the benchmark ages (its
"fresh" window closes as models retrain). This is the idea behind LiveBench, LiveCodeBench, and
time-split SWE-bench.

**S6 (example — "extract PO number from a vendor email"):**
- *Test set:* 200 real vendor emails, PO numbers labelled by two annotators, held out, never
  shown to prompt-tuning.
- *Scoring:* exact string match on the PO number; report precision/recall separately.
- *Chance floor:* ~0 (free-form extraction, not multiple choice).
- *Gaming:* a model could learn "the PO number is the first 8-digit string" — works on the test
  distribution, fails when a vendor changes format. Mitigate with out-of-distribution emails in
  a second hidden slice.

### Answer key
1. `(0.45 − 0.25) / 0.75 ≈ 0.27`. Report it because the raw 45% is mostly the 25% you get for
   free; the model's actual signal is ~27% of the available headroom.
2. The probability that at least one of 10 sampled completions passes all tests. It's
   systematically higher than pass@1 (more shots), so comparing your pass@10 to their pass@1
   is comparing different products; normalise to the same k.
3. Any two: implausibly high scores on old/public benchmarks; large drop on paraphrased or
   freshly-written variants; benchmark score not tracking real downstream performance;
   verbatim benchmark text found in training data / canary triggers.
4. The 71% is a mean over 57 subjects; the two models can have opposite per-category profiles
   (STEM-strong vs balanced). If your task is one weak category, the averages are irrelevant.
5. Prompt format / evaluation harness differences (option labels, answer parsing,
   few-shot vs zero-shot, chain-of-thought or not) — these move scores more than real
   capability gaps between adjacent models.
6. "When a measure becomes a target, it ceases to be a good measure." Example: models trained
   and checkpoint-selected on MMLU-like data saturate MMLU (~90%) while real reasoning gaps
   persist, forcing the field to new benchmarks (MMLU-Pro, GPQA).
7. To narrow a shortlist to 2–3 candidates and flag category weaknesses — never to make the
   final choice. That requires a task-specific eval on your own data and success criteria.